<a href="https://colab.research.google.com/github/sizormohanty/Data-Science/blob/master/Python%20Maverik_Stores_in_Utah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Scrape Maverik Store Zip Codes

This code will attempt to scrape zip codes from a Maverik store locator page. Please note that web scraping can be sensitive to changes in a website's structure. You might need to inspect the actual Maverik store locator page's HTML to adjust the URL and the parsing logic (`find` methods and CSS selectors) if the code doesn't work as expected.

*   **Disclaimer**: Be aware of the website's terms of service regarding web scraping before running this code. Excessive or unauthorized scraping can lead to your IP being blocked.

In [1]:
# Install necessary libraries if you haven't already
!pip install requests beautifulsoup4 pandas selenium webdriver_manager
!apt-get update
!apt install -y google-chrome-stable

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re # For regular expressions to validate/extract zip codes

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from webdriver_manager.core.utils import ChromeType # Import ChromeType
import time # To add a small delay if needed for dynamic content to load

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,863 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [38.8 kB]

In [2]:
# Define the URL for Maverik's store locator page
# You will need to replace this with the actual URL if it's different.
# A common pattern is that store locators often load content dynamically with JavaScript.
# If this is the case, a simple 'requests' call might not get all the data.
# You might need to investigate network requests in your browser's developer tools
# or consider using a tool like Selenium for dynamic content.

# Placeholder URL - PLEASE UPDATE THIS WITH THE ACTUAL MAVERIK STORE LOCATOR URL
url = 'https://www.maverik.com/locations'

# Set up Chrome options for headless browsing
chrome_options = Options()
chrome_options.add_argument("--headless") # Run in headless mode (no visible browser UI)
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/109.0.0.0 Safari/537.36")
chrome_options.binary_location = '/usr/bin/google-chrome-stable'

# Initialize the WebDriver
driver = None
try:
    # Explicitly specify ChromeType.GOOGLE
    service = Service(ChromeDriverManager(chrome_type=ChromeType.GOOGLE).install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    driver.get(url)

    # Give some time for JavaScript to render the page content
    time.sleep(5)

    # Get the page source after JavaScript has executed
    page_content = driver.page_source
    soup = BeautifulSoup(page_content, 'html.parser')
    print("Successfully fetched the page using Selenium. Now attempting to parse.")

except Exception as e:
    print(f"Failed to retrieve the page using Selenium. Error: {e}")
    soup = None # Set soup to None to prevent errors later
finally:
    if driver: # Ensure driver is closed even if an error occurs
        driver.quit()

Failed to retrieve the page using Selenium. Error: Message: unknown error: Chrome failed to start: exited abnormally.
  (unknown error: DevToolsActivePort file doesn't exist)
  (The process started from chrome location /usr/bin/chromium-browser is no longer running, so ChromeDriver is assuming that Chrome has crashed.)
Stacktrace:
#0 0x5a6d51e3d4e3 <unknown>
#1 0x5a6d51b6cc76 <unknown>
#2 0x5a6d51b95d78 <unknown>
#3 0x5a6d51b92029 <unknown>
#4 0x5a6d51bd0ccc <unknown>
#5 0x5a6d51bd047f <unknown>
#6 0x5a6d51bc7de3 <unknown>
#7 0x5a6d51b9d2dd <unknown>
#8 0x5a6d51b9e34e <unknown>
#9 0x5a6d51dfd3e4 <unknown>
#10 0x5a6d51e013d7 <unknown>
#11 0x5a6d51e0bb20 <unknown>
#12 0x5a6d51e02023 <unknown>
#13 0x5a6d51dd01aa <unknown>
#14 0x5a6d51e266b8 <unknown>
#15 0x5a6d51e26847 <unknown>
#16 0x5a6d51e36243 <unknown>
#17 0x7ea6b5c89ac3 <unknown>



In [3]:
zip_codes = []

if soup:
    # This is a generic example. You'll need to inspect the HTML of the Maverik site.
    # Common patterns for finding addresses/zip codes include:
    # - <div class="address">...</div>
    # - <span class="zip-code">...</span>
    # - <p class="store-location">...</p>

    # Let's assume zip codes might appear in text that looks like a zip code (5 digits)
    # within address blocks or general text. Adjust the selector to be more specific.

    # Example 1: Find all elements that might contain address information
    # You might look for specific div classes, for example: `soup.find_all('div', class_='store-address')`
    address_elements = soup.find_all(text=re.compile(r'\b\d{5}\b')) # Find 5-digit numbers

    # Example 2: More targeted approach if you know the HTML structure
    # For instance, if zip codes are inside a span with a specific class:
    # zip_code_spans = soup.find_all('span', class_='store-zip')
    # for span in zip_code_spans:
    #     potential_zip = span.get_text(strip=True)
    #     if re.fullmatch(r'\d{5}', potential_zip):
    #         zip_codes.append(potential_zip)

    for element_text in address_elements:
        # Find all 5-digit numbers in the text
        found_zips = re.findall(r'\b\d{5}(?:-\d{4})?\b', str(element_text))
        for z in found_zips:
            # Basic validation: ensure it's a 5-digit or 5+4 digit zip
            if re.fullmatch(r'\d{5}', z) or re.fullmatch(r'\d{5}-\d{4}', z):
                zip_codes.append(z)

    # Remove duplicates and sort
    zip_codes = sorted(list(set(zip_codes)))

    if not zip_codes:
        print("No zip codes found with the current parsing logic. Please inspect the website's HTML and adjust the `soup.find_all` and regex patterns.")

else:
    print("Could not process, as page retrieval failed or no soup object was created.")

Could not process, as page retrieval failed or no soup object was created.


In [4]:
# Display the extracted zip codes
if zip_codes:
    df_zip_codes = pd.DataFrame(zip_codes, columns=['Maverik_Store_Zip_Code'])
    print(f"Found {len(df_zip_codes)} unique zip codes:")
    display(df_zip_codes)
else:
    print("No zip codes to display. Please re-run the scraping cell after adjusting the parsing logic.")

No zip codes to display. Please re-run the scraping cell after adjusting the parsing logic.
